# UNIVERSIDAD AMAZÓNICA DE PANDO
**Carrera:** Ingeniería de Sistemas  
**Asignatura:** Ingeniería de Software I  
**Proyecto:** Análisis de la distribución poblacional y los indicadores demográficos municipales de Bolivia según el Censo 2024  
**Etapa:** Avance de Proyecto N.º 1 — caracterización, diagnóstico y análisis descriptivo  
**Docente:** Ph. D. Juan Carlos Huanca Guanca  
**Autor:** Cristopher Coaquira  
**R.U. y número de grupo:** completar con los datos correspondientes  
**Lugar y fecha:** Cobija, 21 de octubre de 2026  

**Fuente:** Instituto Nacional de Estadística (INE), POBLACIÓN.xlsx, cuadros 1, 7 y 8. Las cifras representan municipios/TIOC, no personas individuales. El CSV es una extracción derivada. No se infiere causalidad.

## 1. Identificación, objetivo y caracterización
**Dataset:** municipios de Bolivia según tabulados INE del Censo 2024. **Cobertura:** nueve departamentos y 343 unidades municipio/TIOC. **Formato:** CSV derivado del Excel censal. **Objetivo:** caracterizar la población, la edad mediana y el crecimiento intercensal, verificando calidad y distribuciones. Se conserva el archivo original sin modificaciones. La unidad de análisis es municipio/TIOC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

carpeta_exportaciones = Path("exportaciones")
carpeta_exportaciones.mkdir(exist_ok=True)
contador_tablas = 0


def mostrar_y_exportar_tabla(tabla, nombre):
    global contador_tablas
    contador_tablas += 1
    base = f"{contador_tablas:02d}_{nombre}"
    tabla.to_csv(carpeta_exportaciones / f"{base}.csv", encoding="utf-8-sig")
    display(tabla)


p = Path("municipios_bolivia_censo2024_reducido.csv")
df = pd.read_csv(p, encoding="utf-8-sig")
# Todas las tasas porcentuales deben ser numéricas y finitas.
columnas_porcentaje = ["crecimiento_relativo_pct_2012_2024", "tasa_anual_pct_2012_2024", "porcentaje_mujeres_2024"]
for columna in columnas_porcentaje:
    convertido = pd.to_numeric(df[columna], errors="coerce")
    invalidos = convertido.isna() & df[columna].notna()
    if invalidos.any():
        raise ValueError(f"Datos no numéricos en {columna}; filas CSV: {(df.index[invalidos] + 2).tolist()}. Revisar el archivo original.")
    df[columna] = convertido
# Validar crecimiento relativo contra los conteos censales 2012 y 2024.
crecimiento_calculado = (df["poblacion_2024"] / df["poblacion_2012"] - 1) * 100
assert np.allclose(df["crecimiento_relativo_pct_2012_2024"], crecimiento_calculado, atol=1e-7), "Diferencia entre población y tasa relativa"
print("Tipos porcentuales verificados:", df[columnas_porcentaje].dtypes.to_dict())
print(df.shape)
mostrar_y_exportar_tabla(df.head(), "muestra_datos")

## 2. Diccionario de variables
Se describen todos los campos con un valor efectivamente presente, tipo de Pandas, clasificación analítica y rol. Los años forman parte de los nombres de indicadores y no son fechas. Las variables cuantitativas no se confunden con identificadores textuales.

In [ ]:
diccionario = {
    "departamento": ("Departamento al que pertenece el municipio", "Categórica nominal", "Agrupación"),
    "provincia": ("Provincia dentro del departamento", "Categórica nominal", "Agrupación"),
    "municipio": ("Nombre del municipio/TIOC", "Categórica nominal", "Identificación"),
    "poblacion_2001": ("Habitantes censados en 2001", "Cuantitativa discreta", "Medición"),
    "poblacion_2012": ("Habitantes censados en 2012", "Cuantitativa discreta", "Medición"),
    "poblacion_2024": ("Habitantes censados en 2024", "Cuantitativa discreta", "Resultado/medición"),
    "hombres_2024": ("Hombres censados en 2024", "Cuantitativa discreta", "Medición"),
    "mujeres_2024": ("Mujeres censadas en 2024", "Cuantitativa discreta", "Medición"),
    "edad_mediana_2024": ("Edad mediana de la población municipal, años", "Cuantitativa", "Medición"),
    "crecimiento_absoluto_2012_2024": ("Diferencia absoluta de habitantes, 2012 a 2024", "Cuantitativa discreta",
                                       "Medición"),
    "crecimiento_relativo_pct_2012_2024": ("Variación porcentual entre censos", "Cuantitativa continua", "Medición"),
    "tasa_anual_pct_2012_2024": ("Tasa media anual intercensal, porcentaje", "Cuantitativa continua", "Medición"),
    "porcentaje_mujeres_2024": ("Porcentaje de mujeres en 2024", "Cuantitativa continua", "Medición")
}
tabla_diccionario = pd.DataFrame([
    {"variable": c, "significado": diccionario[c][0], "ejemplo_real": str(df[c].iloc[0]),
     "tipo_pandas": str(df[c].dtype), "tipo_analitico": diccionario[c][1], "rol": diccionario[c][2]}
    for c in df.columns])
mostrar_y_exportar_tabla(tabla_diccionario.set_index("variable"), "diccionario_variables")

## 3. Diagnóstico inicial de calidad
La tabla audita todas las variables: tipo reconocido, valores faltantes, porcentajes de nulos, cardinalidad y porcentaje de unicidad. Se verifican duplicados exactos, duplicados por clave territorial compuesta y consistencia por sexo; no se eliminan ni imputan registros. **Hallazgos:** (1) ausencia de nulos en la extracción seleccionada; (2) clave territorial sin duplicados; (3) totales por sexo consistentes. Estas comprobaciones no certifican ausencia de errores en todo el libro original.

In [ ]:
auditoria = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True),
    "porcentaje_unicos": (df.nunique(dropna=True) / len(df) * 100).round(2)
})
mostrar_y_exportar_tabla(auditoria, "auditoria_13_variables")
print("Duplicados exactos:", int(df.duplicated().sum()))
print("Duplicados de clave territorial:", int(df.duplicated(["departamento", "provincia", "municipio"]).sum()))
print("Población total de 2024:", int(df.poblacion_2024.sum()))
print("Filas consistentes por sexo:", int((df.poblacion_2024 == df.hombres_2024 + df.mujeres_2024).sum()), "de",
      len(df))
assert df.shape == (343, 13) and df.poblacion_2024.sum() == 11365333

## 4. Estadística descriptiva de tres variables cuantitativas y dos categóricas
Se examinan población municipal, edad mediana y tasa anual de crecimiento (n válido, media, mediana, desviación estándar muestral, mínimo, máximo, Q1, Q3). Para departamento y provincia se obtienen frecuencias absolutas, porcentajes y moda. Se utiliza clave compuesta departamento–provincia para evitar confundir nombres homónimos.

In [ ]:
numericas = ["poblacion_2024", "edad_mediana_2024", "tasa_anual_pct_2012_2024"]
resumen = df[numericas].agg(["count", "mean", "median", "std", "min", "max"]).T
resumen["Q1"] = df[numericas].quantile(.25)
resumen["Q3"] = df[numericas].quantile(.75)
mostrar_y_exportar_tabla(resumen.round(4), "estadistica_tres_variables")
departamentos = df["departamento"].value_counts().rename_axis("departamento").to_frame("frecuencia")
departamentos["porcentaje"] = (departamentos.frecuencia / len(df) * 100).round(2)
mostrar_y_exportar_tabla(departamentos, "frecuencias_departamento")
provincias = (df.groupby(["departamento", "provincia"]).size().sort_values(ascending=False).to_frame("frecuencia"))
provincias["porcentaje"] = (provincias.frecuencia / len(df) * 100).round(2)
mostrar_y_exportar_tabla(provincias, "frecuencias_provincia")
print("Moda departamento:", departamentos.index[0], int(departamentos.frecuencia.iloc[0]))
print("Moda provincia (frecuencia máxima):", int(provincias.frecuencia.max()))
print("Provincias con frecuencia modal:",
      provincias[provincias.frecuencia == provincias.frecuencia.max()].index.tolist())
print("Total pares departamento-provincia:", len(provincias))
print("Interpretación 1: la media poblacional supera la mediana: asimetría a la derecha.")
print("Interpretación 2: el rango poblacional contiene municipios de tamaños muy distintos.")
print("Interpretación 3: la mitad central de edades medianas se localiza entre Q1 y Q3.")
print("Interpretación 4: existen municipios con crecimiento anual negativo y positivo.")
print("Interpretación 5: el departamento modal no necesariamente concentra la mayor población.")

## 5. Distribuciones e interpretación
**Numérica 1 — población:** la comparación entre mediana, cuartiles y máximo permite examinar asimetría y valores extremos. **Numérica 2 — edad mediana:** el intervalo intercuartílico muestra la concentración de edades medianas municipales. **Categórica 1 — departamento:** se comparan frecuencias, porcentajes y categorías extremas. **Categórica 2 — provincia:** se distinguen las agrupaciones departamento–provincia más y menos frecuentes. Se informa siempre el denominador de 343 municipios/TIOC.

In [ ]:
for v in numericas[:2]:
    s = df[v].dropna()
    print(v, "n=", len(s), "mín=", s.min(), "Q1=", s.quantile(.25), "mediana=", s.median(), "Q3=", s.quantile(.75),
          "máx=", s.max())
for nombre, t in [("departamento", departamentos), ("provincia", provincias)]:
    print(nombre, "máx. frecuencia=", t.frecuencia.max(), "mín. frecuencia=", t.frecuencia.min(), "porcentaje máximo=",
          round(t.porcentaje.max(), 2), "%", "porcentaje mínimo=", round(t.porcentaje.min(), 2), "%")

## 6. Visualizaciones iniciales
Se elaboran **exactamente dos histogramas**, **dos gráficos de barras de frecuencias categóricas** y **un diagrama de caja**. Cada gráfico usa unidades, título y fuente, se guarda en `exportaciones/` y acompaña observaciones numéricas calculadas. Los valores extremos solo se identifican para examen posterior; no se eliminan.

### Figura 1. Histograma de población 2024
La media y la mediana indican asimetría positiva; el máximo es muy superior al cuartil superior. Los números se imprimen junto al gráfico.

In [ ]:
s = df["poblacion_2024"]
plt.figure(figsize=(9, 4.8));
plt.hist(s, bins=30, edgecolor="white");
plt.title("Figura 1. Distribución de población municipal, 2024");
plt.xlabel("Habitantes por municipio/TIOC");
plt.ylabel("Cantidad de municipios/TIOC");
plt.tight_layout();
plt.savefig(carpeta_exportaciones / "grafico_01_poblacion.png", dpi=180);
plt.show()
print(f"Media: {s.mean():,.2f}; mediana: {s.median():,.0f}; Q3: {s.quantile(.75):,.1f}; máximo: {s.max():,.0f}")

### Figura 2. Histograma de edad mediana
Se interpretan la mediana de las edades medianas municipales y la amplitud de la mitad central; los números corresponden a municipios, no a individuos.

In [ ]:
s = df["edad_mediana_2024"];
plt.figure(figsize=(9, 4.8));
plt.hist(s, bins=20, edgecolor="white");
plt.title("Figura 2. Edad mediana municipal, Censo 2024");
plt.xlabel("Edad mediana (años)");
plt.ylabel("Cantidad de municipios/TIOC");
plt.tight_layout();
plt.savefig(carpeta_exportaciones / "grafico_02_edad_mediana.png", dpi=180);
plt.show()
print(
    f"Mediana: {s.median():,.1f} años; Q1: {s.quantile(.25):,.1f}; Q3: {s.quantile(.75):,.1f}; rango: {s.min():,.1f} a {s.max():,.1f}")

### Figura 3. Barras de municipios/TIOC por departamento
Las barras reflejan **frecuencia de municipios**, no número de habitantes. Se identifican máximos y mínimos y su porcentaje del total.

In [ ]:
s = departamentos.frecuencia.sort_values();
plt.figure(figsize=(9, 5));
s.plot.barh();
plt.title("Figura 3. Municipios/TIOC por departamento");
plt.xlabel("Cantidad de municipios/TIOC");
plt.ylabel("Departamento");
plt.tight_layout();
plt.savefig(carpeta_exportaciones / "grafico_03_departamentos.png", dpi=180);
plt.show()
print("Mayor:", departamentos.index[0], int(departamentos.frecuencia.iloc[0]),
      f"({departamentos.porcentaje.iloc[0]:.2f} %)");
print("Menor:", departamentos.index[-1], int(departamentos.frecuencia.iloc[-1]),
      f"({departamentos.porcentaje.iloc[-1]:.2f} %)")

### Figura 4. Barras de frecuencia por provincia
Se representan las **15 combinaciones departamento–provincia con mayor frecuencia**, no la suma de habitantes. La tabla completa de las provincias se encuentra en el apartado 4.

In [ ]:
top = provincias.head(15).copy()
etiquetas = [f"{d} — {p}" for d, p in top.index]
plt.figure(figsize=(11, 7));
plt.barh(etiquetas[::-1], top.frecuencia.to_numpy()[::-1]);
plt.title("Figura 4. Municipios/TIOC por provincia (15 grupos más frecuentes)");
plt.xlabel("Cantidad de municipios/TIOC");
plt.ylabel("Departamento — provincia");
plt.tight_layout();
plt.savefig(carpeta_exportaciones / "grafico_04_provincias.png", dpi=180);
plt.show()
print("Agrupaciones provinciales:", len(provincias));
print("Frecuencia provincial máxima:", int(provincias.frecuencia.max()), "mínima:", int(provincias.frecuencia.min()))

### Figura 5. Boxplot de tasa anual de crecimiento
La mediana y los cuartiles resumen el centro y la dispersión; los puntos aislados indican valores potencialmente atípicos, sin justificar su eliminación.

In [ ]:
s = df["tasa_anual_pct_2012_2024"].dropna();
plt.figure(figsize=(9, 3.8));
plt.boxplot(s, orientation="horizontal");
plt.title("Figura 5. Tasa anual de crecimiento municipal 2012–2024");
plt.xlabel("Variación porcentual anual (%)");
plt.yticks([1], ["Municipios/TIOC"]);
plt.tight_layout();
plt.savefig(carpeta_exportaciones / "grafico_05_boxplot_crecimiento.png", dpi=180);
plt.show()
print(
    f"Mediana: {s.median():.4f} %; Q1: {s.quantile(.25):.4f} %; Q3: {s.quantile(.75):.4f} %; mínimo: {s.min():.4f} %; máximo: {s.max():.4f} %")

## 7. Cinco preguntas analíticas y plan de respuesta
Dos descriptivas, dos comparativas y una relacional. Las preguntas comparativas y relacional se formulan como plan para etapas posteriores; una correlación calculada de manera exploratoria no demuestra causalidad.

In [ ]:
plan = pd.DataFrame([
    ["Descriptiva", "¿Cómo se distribuye la población municipal en 2024?", "poblacion_2024",
     "Descriptivos e histograma"],
    ["Descriptiva", "¿Cómo se distribuye la edad mediana municipal?", "edad_mediana_2024", "Cuartiles e histograma"],
    ["Comparativa", "¿Cómo varía la cantidad de municipios por departamento?", "departamento",
     "Frecuencias, porcentajes y barras"],
    ["Comparativa", "¿Cómo difiere la población total entre departamentos?", "departamento; poblacion_2024",
     "Suma por grupo, contraste de totales"],
    ["Relacional", "¿Existe asociación entre edad mediana y crecimiento anual?",
     "edad_mediana_2024; tasa_anual_pct_2012_2024", "Dispersión y Pearson exploratoria"]],
    columns=["tipo", "pregunta", "variables", "tecnica_prevista"])
mostrar_y_exportar_tabla(plan, "plan_cinco_preguntas")

## 8. Cinco hallazgos, conclusiones y limitaciones
**Hallazgo 1:** la suma de poblaciones municipales es 11.365.333 habitantes. **Hallazgo 2:** la media poblacional municipal supera su mediana, evidencia de asimetría. **Hallazgo 3:** las edades medianas municipales presentan un intervalo intercuartílico mucho más estrecho en años que el rango poblacional en habitantes. **Hallazgo 4:** algunas unidades registran crecimiento intercensal absoluto negativo. **Hallazgo 5:** La Paz concentra más municipios/TIOC que Tarija, pero esto no equivale a la distribución de habitantes.

**Limitación 1:** los agregados municipales no permiten inferir atributos de personas individuales. **Limitación 2:** cambios de límites, cobertura y definiciones requieren cautela en comparaciones intercensales. **Limitación 3:** la selección de 13 campos no representa todas las variables del libro Excel original; los promedios no ponderados dan igual peso a municipios de distinto tamaño.

**Conclusión:** se obtiene una caracterización municipal reproducible y apta para futuros análisis; las asociaciones son exploratorias y no se interpretan causalmente. Las limitaciones de cobertura y agrupación territorial se deben considerar antes de realizar inferencias adicionales.

In [ ]:
print("Hallazgo 1, población total:", int(df.poblacion_2024.sum()))
print("Hallazgo 2, media y mediana:", round(df.poblacion_2024.mean(), 2), df.poblacion_2024.median())
print("Hallazgo 3, cuartiles de edad:", df.edad_mediana_2024.quantile(.25), df.edad_mediana_2024.quantile(.75))
print("Hallazgo 4, crecimiento absoluto negativo:", int((df.crecimiento_absoluto_2012_2024 < 0).sum()))
print("Hallazgo 5, La Paz y Tarija:", int(departamentos.loc["La Paz", "frecuencia"]),
      int(departamentos.loc["Tarija", "frecuencia"]))
print("Correlación exploratoria (no causal):", df[["edad_mediana_2024", "tasa_anual_pct_2012_2024"]].corr().iloc[0, 1])

## 9. Reproducibilidad y fuente
Colocar `municipios_bolivia_censo2024_reducido.csv` en el mismo directorio que este notebook, instalar `pandas`, `numpy`, `matplotlib`, `ipython` y ejecutar en orden de arriba a abajo. Las tablas y cinco imágenes se exportan a `exportaciones/`. Mantener disponible el archivo institucional original `POBLACIÓN.xlsx` o su enlace verificable. **Fuente:** Instituto Nacional de Estadística (INE), Censo de Población y Vivienda 2024, tabulados por municipio/TIOC, cuadros 1, 7 y 8. Se incluyen las unidades y se evita inferencia causal.